In [9]:
import ee
import geemap

In [10]:
ee.Authenticate()
ee.Initialize()

print("Earth Engine initialized successfully!")
print(f"EE version: {ee.__version__}")

Earth Engine initialized successfully!
EE version: 1.6.10


In [11]:
# Create an interactive map
Map = geemap.Map(center=[58.5, -118.5], zoom=8)  # Alberta fire region

# Add a basemap
Map.add_basemap('SATELLITE')

# Display the map
Map

Map(center=[58.5, -118.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [12]:
aoi = ee.Geometry.Rectangle([-120.5, 58.5, -118.0, 60.5])

In [13]:
pre_fire_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2023-04-01', '2023-04-30')  # Before fire season
    .filterBounds(aoi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  # Less than 20% clouds
)

In [14]:
print(f"Pre-fire images found: {pre_fire_collection.size().getInfo()}")

Pre-fire images found: 45


In [15]:
pre_fire_image = pre_fire_collection.median()

In [16]:
post_fire_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2023-09-01', '2023-09-30')  # After fire containment
    .filterBounds(aoi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

print(f"Post-fire images found: {post_fire_collection.size().getInfo()}")

post_fire_image = post_fire_collection.median()

Post-fire images found: 84


In [17]:
def calculate_nbr(image):
    """
    Calculate NBR using NIR (B8) and SWIR (B12) bands
    NBR = (NIR - SWIR) / (NIR + SWIR)
    """
    nir = image.select('B8')
    swir = image.select('B12')
    nbr = nir.subtract(swir).divide(nir.add(swir)).rename('NBR')
    return nbr

In [18]:
nbr_pre = calculate_nbr(pre_fire_image)
nbr_post = calculate_nbr(post_fire_image)

In [19]:
dnbr = nbr_pre.subtract(nbr_post).rename('dNBR')

In [20]:
dnbr_normalized = dnbr.subtract(-0.1).divide(0.9).clamp(0, 1).rename('severity')

In [21]:
# Create interactive map
Map = geemap.Map(center=[59.2, -119.0], zoom=9)

# Visualization parameters
rgb_vis = {
    'bands': ['B4', 'B3', 'B2'],  # Red, Green, Blue
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}

dnbr_vis = {
    'min': -0.1,
    'max': 0.8,
    'palette': ['green', 'yellow', 'orange', 'red', 'darkred']
}

severity_vis = {
    'min': 0,
    'max': 1,
    'palette': ['#2E7D32', '#66BB6A', '#FDD835', '#FB8C00', '#E65100', '#BF360C']
}

# Add layers to map
Map.addLayer(pre_fire_image, rgb_vis, 'Pre-fire RGB (April 2023)')
Map.addLayer(post_fire_image, rgb_vis, 'Post-fire RGB (Sept 2023)')
Map.addLayer(dnbr, dnbr_vis, 'dNBR (raw)')
Map.addLayer(dnbr_normalized, severity_vis, 'Burn Severity (0-1)')
Map.addLayer(aoi, {}, 'Area of Interest', False)

# Add layer control
Map.addLayerControl()

# Display map
Map

Map(center=[59.2, -119.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [22]:
# =============================================================================
# Calculate statistics for the burned area
# =============================================================================

# Get severity statistics
severity_stats = dnbr_normalized.reduceRegion(
    reducer=ee.Reducer.mean().combine(
        reducer2=ee.Reducer.minMax(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.stdDev(),
        sharedInputs=True
    ),
    geometry=aoi,
    scale=10,  # Sentinel-2 resolution
    maxPixels=1e13
)

stats = severity_stats.getInfo()
print("\n📊 Burn Severity Statistics:")
print(f"Mean severity: {stats.get('severity_mean', 0):.3f}")
print(f"Min severity: {stats.get('severity_min', 0):.3f}")
print(f"Max severity: {stats.get('severity_max', 0):.3f}")
print(f"Std Dev: {stats.get('severity_stdDev', 0):.3f}")

# Calculate area of different severity classes
def calculate_severity_areas(severity_image, aoi):
    """Calculate area (in hectares) for each severity class"""
    
    # Define severity thresholds (USGS classification)
    unburned = severity_image.lt(0.1)
    low = severity_image.gte(0.1).And(severity_image.lt(0.27))
    moderate_low = severity_image.gte(0.27).And(severity_image.lt(0.44))
    moderate_high = severity_image.gte(0.44).And(severity_image.lt(0.66))
    high = severity_image.gte(0.66)
    
    # Calculate pixel areas (in square meters)
    pixel_area = ee.Image.pixelArea()
    
    # Calculate areas for each class
    classes = {
        'Unburned': unburned,
        'Low Severity': low,
        'Moderate-Low Severity': moderate_low,
        'Moderate-High Severity': moderate_high,
        'High Severity': high
    }
    
    results = {}
    for name, mask in classes.items():
        area = pixel_area.updateMask(mask).reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=aoi,
            scale=10,
            maxPixels=1e13
        )
        # Convert to hectares
        area_ha = ee.Number(area.get('area')).divide(10000)
        results[name] = area_ha.getInfo()
    
    return results

areas = calculate_severity_areas(dnbr_normalized, aoi)

print("\n🔥 Burned Area by Severity Class:")
for severity_class, area in areas.items():
    print(f"{severity_class:25s}: {area:>10,.0f} hectares")

total_burned = sum([v for k, v in areas.items() if 'Severity' in k])
print(f"{'Total Burned Area':25s}: {total_burned:>10,.0f} hectares")


📊 Burn Severity Statistics:
Mean severity: 0.478
Min severity: 0.000
Max severity: 1.000
Std Dev: 0.318

🔥 Burned Area by Severity Class:
Unburned                 :    406,915 hectares
Low Severity             :    607,562 hectares
Moderate-Low Severity    :    579,631 hectares
Moderate-High Severity   :    627,912 hectares
High Severity            :    932,148 hectares
Total Burned Area        :  2,747,252 hectares
